In [1]:
print()
import cv2
from utils import preprocess_image,get_gradcam
import os
from tensorflow.keras.models import load_model
from tf_explain.core.integrated_gradients import IntegratedGradients

In [2]:
model=load_model("C:\\Users\\PratushPc\\OneDrive\\Desktop\\ExCV\\models\\best_model.keras")

In [3]:
import numpy as np

def saliency_entropy(saliency_map):

    saliency = saliency_map.copy()

    saliency = saliency.astype(np.float32)

    saliency = saliency - saliency.min()

    saliency = saliency / (
        saliency.sum() + 1e-10
    )

    entropy = -np.sum(
        saliency *
        np.log(
            saliency + 1e-10
        )
    )

    return entropy


In [4]:
def deletion_curve(
    model,
    img_processed,
    heatmap,
    pred_idx,
    percentages=np.arange(0, 1.1, 0.1)
):
    """
    img_processed : (1,224,224,3) preprocessed image
    heatmap       : (224,224)
    pred_idx      : class being explained
    """

    # Flatten saliency map
    saliency = heatmap.flatten()

    # Most important -> least important
    sorted_idx = np.argsort(saliency)[::-1]

    total_pixels = len(sorted_idx)

    confidences = []
    
    original_img = img_processed[0].copy()

    for p in percentages:

        img_mod = original_img.copy()
        # print(img_mod.shape)

        num_pixels = int(total_pixels * p)

        if num_pixels > 0:

            pixels_to_delete = sorted_idx[:num_pixels]

            rows = pixels_to_delete // 224
            cols = pixels_to_delete % 224

            img_mod[rows, cols, :] = 0

        pred = model.predict(
            np.expand_dims(img_mod,axis=0),
            verbose=0
        )[0]
        
        
        confidences.append(pred[pred_idx])

    return percentages, confidences

In [5]:
def compute_aopc(confidences):
    
    original = confidences[0]

    drops = [
        original - c
        for c in confidences
    ]

    return np.mean(drops)

In [6]:
TEST_PATH='C:/Users/PratushPc/OneDrive/Desktop/ExCV/data/COVID_19_dataset/test/'

In [7]:
gradcam_entropies=[]

x_batch=[]
y_batch=[]
gradcam_batch=[]


for class_ in os.listdir(TEST_PATH)[1:]:
    count=30
    for img_name in os.listdir(TEST_PATH+class_):
        img_path=TEST_PATH+class_+"/"+img_name
        img,processed_img=preprocess_image(img_path)
        x_batch.append(processed_img[0])
        
        pred=model.predict(processed_img,verbose=0)[0]
        pred_class=np.argmax(pred)
        y_batch.append(pred_class)
        
        cam=get_gradcam(model,img_path,pred_class)
        
        gradcam_batch.append(cam)
        
        gradcam_entropies.append(saliency_entropy(cam))
        
        count-=1
        
        if not count:
            break

In [8]:
print(f"GradCam Mean: {np.mean(gradcam_entropies)}")
print()
print(f"GradCam Std: {np.std(gradcam_entropies)}")

GradCam Mean: 10.03579330444336

GradCam Std: 0.39342594146728516


In [9]:
x_batch=np.array(x_batch)
y_batch=np.array(y_batch)
gradcam_batch=np.array(gradcam_batch)

In [10]:
print(x_batch.shape)
print(y_batch.shape)
print(gradcam_batch.shape)

(90, 224, 224, 3)
(90,)
(90, 224, 224)


In [11]:
from sklearn.metrics import auc
gradcam_auc = []

gradcam_aopc = []

for i in range(len(x_batch)):

    pred = model.predict(
        x_batch[i:i+1],
        verbose=0
    )[0]
    
    pred_idx = np.argmax(pred)

    p, conf = deletion_curve(
        model,
        x_batch[i:i+1],
        gradcam_batch[i],
        pred_idx
    )
    
    gradcam_aopc.append(compute_aopc(conf))

    gradcam_auc.append(
        auc(p, conf)
    )


In [12]:
print("GradCAM Mean AUC:", np.mean(gradcam_auc))
print()
print("GradCAM AUC Std:", np.std(gradcam_auc))

GradCAM Mean AUC: 0.4474964535484711

GradCAM AUC Std: 0.37839153011615473


In [13]:
print("GradCAM AOPC:",np.mean(gradcam_aopc))

GradCAM AOPC: 0.5002671
